# Global model for cold-start personalization: all week-1 code in one notebook (Person 2)
Everything from Phases 1 to 5, in order, with headings and detailed comments. Run top to bottom (Runtime > Run all).

| Section | What it does | Key output |
|---|---|---|
| 0 | Load the data | shapes |
| 1 | Split votes by session and check it matches Person 1's file | PHASE 1: PASS |
| 2 | Turn each response's text into 6 numbers; save them | PHASE 2: PASS |
| 3 | Interface spec (written separately in `interface_spec.md`) | none |
| 4 | Global model tools and tests | 3 tests PASS |
| 5 | Tune lambda on validation, compare models, check caveats, figures | REPORT A |
| 5B | Evaluate the test split ONCE and save `global.json` | REPORT B (locked) |

**Rules**
- The test split is touched only in Section 5B, once, after lambda is chosen on validation.
- Section 5B is locked by `CONFIRMED = False`, so Run all stops there safely.
- Raw text lives in `clean_en.parquet`. Never commit it. Everything saved to `results/` is numbers only.
- Section 2 recomputes features (a few minutes). Set `RECOMPUTE_FEATURES = False` in Section 0 to reuse the saved file.

# Section 0: Setup and load the data

In [ ]:
# drive.mount connects this Colab session to your Google Drive. A popup asks for permission the first time.
from google.colab import drive
drive.mount("/content/drive")

import re                                   # regular expressions: patterns for finding text (headers, bullets, bold)
import json, datetime                       # json: save results; datetime: stamp today's date
import numpy as np                          # numpy: fast math on arrays of numbers
import pandas as pd                         # pandas: tables ("DataFrames")
from pathlib import Path                    # Path: builds file paths safely
from sklearn.model_selection import train_test_split    # the same random splitter Person 1 used
from sklearn.linear_model import LogisticRegression     # used ONLY to double-check my own fit (tests in Section 4)

# Where the input files live (D) and where outputs go (OUT). If your folders are named differently,
# edit only the part after "MyDrive/".
D = Path("/content/drive/MyDrive/arena/data")
OUT = Path("/content/drive/MyDrive/arena/results")
OUT.mkdir(parents=True, exist_ok=True)      # create the results folder if missing; do nothing if it exists

RECOMPUTE_FEATURES = True    # True: rebuild features from the text (a few minutes). False: reuse features_raw.parquet.

df = pd.read_parquet(D / "clean_en.parquet")   # the votes: one row per vote, with the prompt and both responses (text)
sp = pd.read_parquet(D / "splits.parquet")     # Person 1's split labels: one row per vote (id, session, split)
print(df.shape, sp.shape)   # shape = (rows, columns). Expect: (71102, 16) (71102, 3)

# Section 1: Lock the split (Phase 1)
Why: votes from one session are similar. If a session is in both train and test, the model effectively sees test answers during training (leakage).

In [ ]:
def split_by_session(df, seed=42):
    """Split votes 70/15/15 into train/validation/test BY SESSION (not by vote).
    Same call and seed as Person 1, so the result should match their file exactly."""
    # unique() lists each session id once. We split the sessions, not the votes,
    # so all votes of one session end up in the same split.
    sessions = df["evaluation_session_id"].unique()
    # First cut: 70% of sessions to train, 30% held out. random_state=seed makes the shuffle repeatable.
    train, hold = train_test_split(sessions, test_size=0.30, random_state=seed)
    # Second cut: split the held-out 30% in half: 15% validation, 15% test.
    val, test = train_test_split(hold, test_size=0.50, random_state=seed)
    out = df[["id", "evaluation_session_id"]].copy()                       # one row per vote
    out["split"] = "test"                                                  # default label
    out.loc[out.evaluation_session_id.isin(train), "split"] = "train"      # votes whose session is in `train`
    out.loc[out.evaluation_session_id.isin(val), "split"] = "validation"   # votes whose session is in `val`
    return out

mine = split_by_session(df)                 # my version of the split
# Line up my labels with Person 1's labels, vote by vote (matching on the vote id).
cmp = mine.merge(sp[["id", "split"]], on="id", suffixes=("_mine", "_p1"))

p1 = {   # every check in one dictionary so it prints cleanly
    "votes_per_split": mine.split.value_counts().to_dict(),
    "sessions_per_split": mine.groupby("split").evaluation_session_id.nunique().to_dict(),
    # For each session, count how many different splits it appears in. The max must be 1.
    "max_splits_per_session": int(mine.groupby("evaluation_session_id").split.nunique().max()),
    "votes_compared": len(cmp),
    "mismatches_vs_person1": int((cmp.split_mine != cmp.split_p1).sum()),                 # must be 0
    "ids_identical": bool(sp.id.is_unique and set(sp.id) == set(df.id)),                  # same ids in both files
}
p1_pass = (p1["max_splits_per_session"] == 1 and p1["mismatches_vs_person1"] == 0
           and p1["ids_identical"] and p1["votes_compared"] == len(df))
for k, v in p1.items():
    print(k, ":", v)
print("PHASE 1:", "PASS" if p1_pass else "FAIL")
# Expect votes: train 49777, test 10681, validation 10644   (your run matched)
# Expect sessions: train 43045, test 9224, validation 9224   (your run matched)

# From here on, Person 1's split labels are the truth. .map looks up each vote id in their table.
df["split"] = df["id"].map(sp.set_index("id")["split"])
print("missing split labels:", int(df.split.isna().sum()))   # Expect 0

# Section 2: Build the features (Phase 2)
The file only has character lengths. The model needs more numbers per response: words, headers, bullets, bold, code blocks, refusal.
- Words count the whole response.
- Headers, bullets and bold are counted **outside code blocks only** (code comments like `# note` are not headers).
- Code blocks and the refusal flag use the whole response.
- The refusal flag is a rule of thumb, not ground truth.

In [ ]:
# ---- Patterns (regular expressions). re.compile builds a pattern once so it can be reused fast. ----
# ^ = start of a line; \s = whitespace; {0,3} = 0 to 3 times; {1,6} = 1 to 6 times.
HEADER = re.compile(r"^\s{0,3}#{1,6}\s")                 # a markdown heading: up to 3 spaces, 1-6 '#' signs, then a space
BULLET = re.compile(r"^\s*(?:[-*+\u2022]|\d+[.)])\s+")   # a list item: starts with -, *, +, a bullet dot, or a number then . or )
BOLD   = re.compile(r"\*\*[^*\n]+\*\*")                  # **bold text** (no '*' or line break inside)
# Refusal phrases such as "I can't help", "I cannot", "I won't", "I'm unable to", "sorry, but I", "as an AI".
# re.I = ignore upper/lower case. \u2019 is the curly apostrophe.
REFUSAL = re.compile(
    r"(i\s*(?:can[\u2019']?t|cannot|won[\u2019']?t)\s+(?:help|assist|provide|do|fulfill|comply|create|generate|write|continue)"
    r"|i[\u2019']?m\s+(?:unable|not able)\s+to|i\s+am\s+(?:unable|not able)\s+to"
    r"|(?:sorry|apologi[sz]e),?\s+but\s+i|as an ai)", re.I)

def strip_code(text):
    """Remove ``` code blocks (and an unclosed trailing block) so code lines are not counted as headers/bullets."""
    # Delete everything between a pair of ``` fences. re.S lets '.' match line breaks too; '.*?' = as little as possible.
    text = re.sub(r"```.*?```", "", text, flags=re.S)
    # If a lone ``` remains (a block that was never closed), drop everything from it to the end.
    i = text.find("```")
    return text if i == -1 else text[:i]

def count_features(text, strip=True):
    """Return (words, headers, bullets, bold, code_blocks) for one response."""
    plain = strip_code(text) if strip else text     # text with code blocks removed (or kept, if strip=False)
    lines = plain.split("\n")                       # headers and bullets are decided line by line
    return (len(text.split()),                              # words: split on whitespace, whole response (code included)
            sum(bool(HEADER.match(l)) for l in lines),      # headers: lines matching the header pattern
            sum(bool(BULLET.match(l)) for l in lines),      # bullets: lines matching the bullet pattern
            len(BOLD.findall(plain)),                       # bold: number of **...** pairs
            text.count("```") // 2)                         # code blocks: fences come in pairs, so count // 2

def refusal_flag(text):
    """1 if a refusal phrase appears in the first 300 characters, else 0 (refusals usually open the reply)."""
    return int(bool(REFUSAL.search(text[:300])))

# Self-test: the "# c" and "- d" lines are inside a code block and must NOT count.
t = "# T\n**b** x\n- one\n1. two\n```py\n# c\n- d\nx\n```"
print(count_features(t), count_features(t, strip=False), refusal_flag("I'm sorry, but I can't help with that."))
# Expect: (15, 1, 2, 1, 1) (15, 2, 3, 1, 1) 1   -> stripped counts first, unstripped second, refusal 1

def feats(series):
    """Turn a column of response texts into a table of numbers (one row per response)."""
    f = pd.DataFrame([count_features(t) for t in series],
                     columns=["words", "headers", "bullets", "bold", "code"], index=series.index)
    f["refusal"] = [refusal_flag(t) for t in series]
    return f

In [ ]:
FEATURES_FILE = OUT / "features_raw.parquet"
if RECOMPUTE_FEATURES or not FEATURES_FILE.exists():
    fa = feats(df.resp_a).add_prefix("a_")     # features of response A: a_words, a_headers, ... (takes a few minutes)
    fb = feats(df.resp_b).add_prefix("b_")     # same for response B
    # Glue together: vote id, session, split, who won, then A's numbers and B's numbers side by side.
    feat = pd.concat([df[["id", "evaluation_session_id", "split", "winner"]], fa, fb], axis=1)
    feat.to_parquet(FEATURES_FILE, index=False)   # numbers only (no text); later sections and Person 1 can reuse it
else:
    feat = pd.read_parquet(FEATURES_FILE)         # reuse the saved numbers
print(feat.shape)   # Expect: (71102, 16)

# Proof the code-block fix worked: headers before vs after, on up to 2000 code-containing responses.
pool = df[feat.a_code > 0]                                   # responses A that contain at least one code block
sample = pool.sample(min(2000, len(pool)), random_state=0)   # random sample (repeatable because of random_state)
hdr_old = sum(count_features(t, strip=False)[1] for t in sample.resp_a)   # headers if code lines are counted
hdr_new = sum(count_features(t, strip=True)[1] for t in sample.resp_a)    # headers outside code blocks only
print("headers in 2000 code-containing responses: counting code =", hdr_old, "| outside code only =", hdr_new)
# Your run: counting code = 10557 | outside code only = 7498

In [ ]:
# ---- Phase 2 report ----
num = [c_ for c_ in feat.columns if c_[:2] in ("a_", "b_")]      # all 12 feature columns
cnt = [c_ for c_ in num if not c_.endswith("refusal")]           # the count columns (not the 0/1 refusal flag)

# Baseline = "always pick the longer response", on decisive votes (A or B won, not tie / both bad).
dec2 = feat[feat.winner.isin(["model_a", "model_b"])].copy()
dec2["a_won"] = dec2.winner == "model_a"
def longer_acc(frame, a, b):
    """Accuracy of 'the response with more (a vs b) wins'. Votes where both are equal are skipped."""
    f = frame[frame[a] != frame[b]]
    return round(float(((f[a] > f[b]) == f.a_won).mean()), 4), len(f)       # (accuracy, number of votes used)
tmp = df.loc[dec2.index]; f = tmp[tmp.len_a != tmp.len_b]
chars_acc = (round(float(((f.len_a > f.len_b) == (f.winner == "model_a")).mean()), 4), len(f))   # Person 1's number
words_all = longer_acc(dec2, "a_words", "b_words")
words_split = {s: longer_acc(dec2[dec2.split == s], "a_words", "b_words") for s in ["train", "validation", "test"]}
a_share = round(float(dec2.a_won.mean()), 4)                    # share of decisive votes where A won (position bias)

print("================ PHASE 1-2 REPORT ================")
print("PHASE 1:", "PASS" if p1_pass else "FAIL")
print("FEATURES shape:", feat.shape, "| NaN total:", int(feat[num].isna().sum().sum()), "| min words:", int(feat[["a_words","b_words"]].min().min()))
print(feat[num].describe(percentiles=[.5, .95, .99]).T[["mean", "50%", "95%", "99%", "max"]].round(2))
print("\nmax / 99th percentile, raw vs log(1+x)  (smaller = tamer tails):")
for c_ in cnt:
    p99 = feat[c_].quantile(.99); mx = feat[c_].max()
    print(f"  {c_:11s} raw {mx / max(p99, 1e-9):6.1f}x   log {np.log1p(mx) / max(np.log1p(p99), 1e-9):5.2f}x")
print("\nlonger-wins by chars (all):", chars_acc, "  Expect (0.6002, 51552)")
print("longer-wins by words (all):", words_all, "  (your run: (0.6006, 51332))")
print("longer-wins by words per split:", words_split)
print("A win share:", a_share, "  (your run: 0.4952)")
ok2 = (int(feat[num].isna().sum().sum()) == 0 and chars_acc == (0.6002, 51552)
       and abs(words_all[0] - 0.60) < 0.01 and 0.48 < a_share < 0.51)
print("PHASE 2:", "PASS" if ok2 else "CHECK")
print("==================================================")

# Section 3: Interface spec (Phase 3)
Written separately in `interface_spec.md` (no code here). It is the contract Person 3 and the cloud track build against:
9 functions (`response_features`, `pair_features`, `scale_dx`, `load_artifacts`, `assign_type`, `init_user`, `update_user`, `predict`, `profile`),
the feature definitions, the artifact files, and the decisions made (log transform, confidence = 1 - d1/d2, scale only with no intercept, no encoder in the cloud).
The code below (Sections 2 and 4) is the first working version of the feature and fitting pieces the spec describes.

# Section 4: The global model and its tests (Phase 4)
- Model: `P(A preferred) = sigmoid(w . dx)`, where `dx = features(A) - features(B)` after `log(1 + count)` and scaling.
- Trained on both A/B orders (every vote plus its mirror copy), so no intercept is needed.
- `y = 1` if A won, 0 if B won. Ties and both-bad votes are left out (decisive votes only).

In [ ]:
NAMES = ["words", "headers", "bullets", "bold", "code", "refusal"]                         # the 6 raw feature names
LOGNAMES = ["log_words", "log_headers", "log_bullets", "log_bold", "log_code", "refusal"]  # names after the log transform

def resp_matrix(f, side, mode):
    """(n_votes x 6) feature array for one side ('a' or 'b'). mode='log' applies log(1+count) to the 5 counts."""
    X = f[[f"{side}_{n}" for n in NAMES]].to_numpy(dtype=float).copy()   # .copy(): make it writable
    if mode == "log":
        X[:, :5] = np.log1p(X[:, :5])       # log(1 + count); the +1 keeps a count of 0 from breaking the log
    return X                                # the refusal flag (column 6) stays 0/1

def make_dx(f, mode="log", cols=None):
    """dx = features(A) - features(B) for every vote. cols picks a subset of features (None = all 6)."""
    dx = resp_matrix(f, "a", mode) - resp_matrix(f, "b", mode)
    return dx if cols is None else dx[:, cols]

def sigmoid(z):
    """Turns any number into a probability between 0 and 1 (clip avoids overflow)."""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def fit_logistic(dx, y, w_prior, lam, sw=None, w0=None, max_iter=50, tol=1e-9):
    """Find weights w minimizing:  sum_i sw_i * logloss_i  +  lam * ||w - w_prior||^2   (Newton's method).
    - Newton's method: repeatedly use the slope (gradient) and curvature (Hessian) of the loss to jump close to the minimum.
    - sw: optional per-vote weights (used by the bootstrap; None = every vote counts once).
    - w0: optional starting point (a warm start makes bootstrap refits fast; None = start at w_prior).
    - No intercept (decision D1). lam must be > 0. w_prior = 0 for the global model."""
    assert lam > 0
    sw = np.ones(len(y)) if sw is None else sw
    w = np.array(w_prior if w0 is None else w0, dtype=float)
    d = dx.shape[1]
    for _ in range(max_iter):
        p = sigmoid(dx @ w)                                              # current predicted P(A wins)
        grad = dx.T @ (sw * (p - y)) + 2.0 * lam * (w - w_prior)         # slope of the loss
        hess = (dx * (sw * p * (1 - p))[:, None]).T @ dx + 2.0 * lam * np.eye(d)   # curvature of the loss
        step = np.linalg.solve(hess, grad)                               # the Newton step
        w = w - step
        if np.max(np.abs(step)) < tol:                                   # converged
            break
    return w

def symmetrize(dx, y):
    """Add a mirror copy of every vote (swap A and B): dx -> -dx and y -> 1 - y. Cancels position bias."""
    return np.vstack([dx, -dx]), np.concatenate([y, 1.0 - y])

# ----- Keep decisive votes only (A or B won) and set up labels and masks -----
dec = feat[feat.winner.isin(["model_a", "model_b"])].reset_index(drop=True)
y_all = (dec.winner == "model_a").to_numpy(dtype=float)        # 1.0 if A won, 0.0 if B won
sp_ = dec.split.to_numpy()
tr, va, te = sp_ == "train", sp_ == "validation", sp_ == "test"   # boolean masks per split
sess = dec.evaluation_session_id.to_numpy()                    # session id per vote (used to resample whole sessions)
a_w = dec.a_words.to_numpy(); b_w = dec.b_words.to_numpy()
neq = a_w != b_w                                               # votes where the word counts differ
base_correct = ((a_w > b_w) == (y_all == 1))                   # baseline "longer wins": was it right?
mask_va, mask_te = va & neq, te & neq                          # the votes the baseline can answer, per split
print("decisive votes: train", int(tr.sum()), "| validation", int(va.sum()), "| test", int(te.sum()))
print("baseline votes (word counts differ): validation", int(mask_va.sum()), "| test", int(mask_te.sum()))
# Expect (your run): train 36151 | validation 7715 | test 7739 ; baseline votes: validation 7665 | test 7693

def build(mode, cols=None):
    """Prepare one model variant: scaled differences for all votes plus the mirrored TRAIN set."""
    dx = make_dx(dec, mode, cols)
    # Scale only: divide each feature by its root mean square on TRAIN votes. No mean is subtracted.
    scale = np.sqrt(np.mean(dx[tr] ** 2, axis=0)); scale[scale == 0] = 1.0     # guard against dividing by 0
    z = dx / scale                                             # scaled differences for ALL votes
    zs, ys = symmetrize(z[tr], y_all[tr])                      # train votes plus mirror copies (used for fitting)
    return dict(z=z, zs=zs, ys=ys, scale=scale)

def fit(model, lam):
    """Fit one variant at regularization strength lam. The prior is all zeros (the global model)."""
    return fit_logistic(model["zs"], model["ys"], np.zeros(model["z"].shape[1]), lam)

def logloss(w, z, y):
    """Average log loss: -log(probability given to the real winner). Lower is better; 50/50 guessing = 0.6931."""
    p = np.clip(sigmoid(z @ w), 1e-15, 1 - 1e-15)              # clip avoids log(0)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

def acc(w, z, y):
    """Share of votes where the predicted winner (p > 0.5 means A) is the real winner."""
    return float(np.mean((sigmoid(z @ w) > 0.5) == (y == 1)))

# Three model variants: log features, raw features (no log), and log words only (one feature).
M = {"log": build("log"), "raw": build("raw"), "words": build("log", cols=[0])}

In [ ]:
# ---- Tests: must all pass before trusting any number ----
LAM1 = 1.0                                   # a first, untuned lambda, used only for these tests
w1 = fit(M["log"], LAM1)
tests = {}

# TEST 1: my fit should equal scikit-learn's well-known implementation. Their C relates to my lam by lam = 1 / (2C).
sk = LogisticRegression(C=1 / (2 * LAM1), fit_intercept=False, tol=1e-10, max_iter=5000)
sk.fit(M["log"]["zs"], M["log"]["ys"].astype(int))          # scikit-learn needs hard 0/1 labels; ours are exactly 0/1
diff = float(np.max(np.abs(sk.coef_[0] - w1)))             # biggest difference between the two weight vectors
tests["fit matches scikit-learn (max weight diff < 1e-4)"] = diff < 1e-4
print("max weight diff vs scikit-learn:", diff)             # your run: about 1e-8

# TEST 2: symmetry. Swapping A and B should give probabilities that add up to exactly 1.
z_chk = M["log"]["z"][va][:1000]
tests["swap symmetry"] = bool(np.allclose(sigmoid(z_chk @ w1) + sigmoid(-z_chk @ w1), 1.0))

# TEST 3 (decision D1): if we ALLOW an intercept, the best one should be 0 on mirror-symmetric data.
sk_i = LogisticRegression(C=1 / (2 * LAM1), fit_intercept=True, tol=1e-10, max_iter=5000)
sk_i.fit(M["log"]["zs"], M["log"]["ys"].astype(int))
b = float(sk_i.intercept_[0])
tests["intercept is about 0 when trained on both orders (|b| < 1e-6)"] = abs(b) < 1e-6
print("fitted intercept:", b)                               # your run: about -6.5e-19

print("first fit at lam=1, validation accuracy / log loss:", round(acc(w1, M["log"]["z"][va], y_all[va]), 4), round(logloss(w1, M["log"]["z"][va], y_all[va]), 4))
for k, v in tests.items():
    print("PASS" if v else "FAIL", "-", k)

# Section 5: Tune lambda and check the caveats (Phase 5, part A: train and validation only)
Rule fixed in advance: lambda = the grid value with the lowest **validation log loss** (accuracy is too noisy to separate values).
The test split is not used in this section.

In [ ]:
# ----- 5.1 Tune lambda on validation (bigger lambda = weights kept smaller) -----
GRID = [0.01, 0.1, 1, 10, 100, 300, 1000, 3000, 10000]   # candidate lambda values. The curve is flat for small values.
tune = {}                                   # results per variant: list of (lam, validation log loss, validation accuracy)
for name in ["log", "raw"]:
    rows_ = []
    for lam in GRID:
        w = fit(M[name], lam)               # fit on TRAIN at this lambda
        # Score on ALL decisive validation votes. The test split is not used here.
        rows_.append((lam, logloss(w, M[name]["z"][va], y_all[va]), acc(w, M[name]["z"][va], y_all[va])))
    tune[name] = rows_
best = {}
for name in tune:
    lls = [r[1] for r in tune[name]]                         # the validation log losses
    best[name] = tune[name][int(np.argmin(lls))][0]          # lambda with the LOWEST validation log loss wins
LAM = best["log"]                                            # the lambda used for the main (log) model
print("chosen lambda: log =", best["log"], "| raw =", best["raw"])    # your run: log = 300 | raw = 0.01
# Log loss is flat across small lambda (about 36,000 votes, only 6 weights), so near-ties can flip the winner.
edge = [n for n in best if best[n] in (GRID[0], GRID[-1])]
print("WARNING: chosen at the edge of the grid for", edge, "(harmless here: the curve is flat)" if edge else "") if edge else print("chosen values are inside the grid")

In [ ]:
# ----- 5.2 Compare models on the SAME validation votes, with session-level bootstrap intervals -----
def per_vote(name, lam, m):
    """Fit a variant at lam, then return its weights plus, for each vote in mask m,
    whether it was correct (1.0/0.0) and its log loss. Used for paired comparisons."""
    w = fit(M[name], lam)
    z = M[name]["z"][m]; y = y_all[m]
    p = sigmoid(z @ w); pc = np.clip(p, 1e-15, 1 - 1e-15)
    return w, ((p > 0.5) == (y == 1)).astype(float), -(y * np.log(pc) + (1 - y) * np.log(1 - pc))

def session_boot(d, codes, B=1000, seed=0):
    """Mean of d with a 95% interval, resampling WHOLE SESSIONS with replacement B times.
    Why sessions: votes inside one session are related, so resampling single votes would make the interval too narrow.
    d: one number per vote (e.g. model correct minus baseline correct). codes: session number (0..m-1) per vote."""
    rng = np.random.default_rng(seed)                           # fixed seed = repeatable
    S = np.bincount(codes, weights=d)                           # sum of d within each session
    N = np.bincount(codes).astype(float)                        # number of votes in each session
    m = len(S)                                                  # number of sessions
    out = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, m, m)                             # draw m sessions with replacement
        out[b] = S[idx].sum() / N[idx].sum()                    # mean of d over the drawn sessions
    lo, hi = np.percentile(out, [2.5, 97.5])                    # middle 95% of the resampled means
    return float(d.mean()), float(lo), float(hi)

codes_va = pd.factorize(sess[mask_va])[0]                       # turn session ids into numbers 0..m-1
w_log, c_log, l_log = per_vote("log", best["log"], mask_va)     # main model (log features)
w_raw, c_raw, l_raw = per_vote("raw", best["raw"], mask_va)     # raw features model
w_wds, c_wds, l_wds = per_vote("words", best["log"], mask_va)   # log words only
c_base = base_correct[mask_va].astype(float)                    # baseline correct? (1.0/0.0)
n_va = int(mask_va.sum())

# For each comparison, an interval that includes 0 means "no clear difference".
cmp_ = {
  "accuracy: log model minus baseline": session_boot(c_log - c_base, codes_va),
  "log loss: log model minus words-only (negative = log model better)": session_boot(l_log - l_wds, codes_va),
  "log loss: log model minus raw model (negative = log model better)": session_boot(l_log - l_raw, codes_va),
}

# ----- 5.3 Are the weights really different from zero? Refit 300 times on resampled training sessions -----
codes_tr = pd.factorize(sess[tr])[0]; m_tr = codes_tr.max() + 1  # session numbers for the training votes
rng = np.random.default_rng(1)
W = []                                                           # will hold 300 weight vectors
w_hat = fit(M["log"], LAM)                                       # the main fit (also the warm start below)
for b in range(300):
    # Draw m_tr sessions with replacement; cnt[s] = how many times session s was drawn (0, 1, 2, ...).
    cnt = np.bincount(rng.integers(0, m_tr, m_tr), minlength=m_tr).astype(float)
    sw = cnt[codes_tr]                                           # per-vote weight = its session's draw count
    # Refit with these weights. The mirrored half of the training data gets the same weights.
    W.append(fit_logistic(M["log"]["zs"], M["log"]["ys"], np.zeros(6), LAM, sw=np.concatenate([sw, sw]), w0=w_hat))
W = np.array(W)
w_lo, w_hi = np.percentile(W, [2.5, 97.5], axis=0)               # 95% interval per weight

# ----- 5.4 Do validation votes repeat training prompts? (needs embedding_index.parquet) -----
# Splitting by session does not stop the same prompt text landing in two splits.
overlap_msg = []                                                 # lines printed in the report
try:
    # Look for the file in arena/data, then in arena. next(..., None) returns None if neither exists.
    ei_path = next((p for p in [D / "embedding_index.parquet", D.parent / "embedding_index.parquet"] if p.exists()), None)
    if ei_path is None:
        overlap_msg.append("  skipped: embedding_index.parquet not found in arena/data or arena")
    else:
        ei = pd.read_parquet(ei_path)                            # maps each vote id to a row of the embeddings (= a distinct prompt)
        if {"id", "emb_row"} <= set(ei.columns):
            row_of = ei.set_index("id")["emb_row"]
            er = dec["id"].map(row_of).to_numpy()                # prompt number for every decisive vote
            seen = set(er[tr][~pd.isna(er[tr])].tolist())        # prompt numbers that occur in TRAIN
            in_train = np.array([(r in seen) for r in er])       # does each vote's prompt also occur in train?
            for label, mm in [("validation votes whose prompt also appears in train", mask_va & in_train),
                              ("validation votes with a prompt not seen in train", mask_va & ~in_train)]:
                if mm.sum() > 0:
                    pv = sigmoid(M["log"]["z"][mm] @ w_log)      # model's P(A wins) for those votes
                    overlap_msg.append(f"  {label}: n={int(mm.sum())}  model acc={float(np.mean((pv > 0.5) == (y_all[mm] == 1))):.4f}  baseline acc={float(base_correct[mm].mean()):.4f}")
        else:
            overlap_msg.append("  skipped: embedding_index columns are " + str(ei.columns.tolist()))
except Exception as e:
    overlap_msg.append("  skipped (" + type(e).__name__ + "): " + str(e)[:120])

In [ ]:
# ----- REPORT A: everything learned from train and validation -----
print("================ REPORT A ================")
print("TUNING on validation (all decisive validation votes), lowest log loss wins:")
for name in ["log", "raw"]:
    print(f"  {name} features:")
    for lam, ll, ac in tune[name]:
        mark = "  <== chosen" if lam == best[name] else ""
        print(f"    lam={lam:>8g}  log loss={ll:.4f}  accuracy={ac:.4f}{mark}")
print()
print(f"COMPARISON on the same {n_va} validation votes (word counts differ):")
print(f"  baseline (longer response)        accuracy {float(c_base.mean()):.4f}")
print(f"  log model (lam={best['log']:g})        accuracy {float(c_log.mean()):.4f}  log loss {float(l_log.mean()):.4f}")
print(f"  raw model (lam={best['raw']:g})        accuracy {float(c_raw.mean()):.4f}  log loss {float(l_raw.mean()):.4f}")
print(f"  log words only (lam={best['log']:g})   accuracy {float(c_wds.mean()):.4f}  log loss {float(l_wds.mean()):.4f}")
print()
print("DIFFERENCES with 95% session-bootstrap intervals (an interval that includes 0 = no clear difference):")
for k, (pt, lo, hi) in cmp_.items():
    print(f"  {k}: {pt:+.4f}  [{lo:+.4f}, {hi:+.4f}]  ->", "clear" if (lo > 0 or hi < 0) else "NOT clear")
print()
print("WEIGHTS of the log model (scaled units) with 95% intervals from 300 refits:")
for n_, w_, lo, hi in zip(LOGNAMES, w_hat, w_lo, w_hi):
    print(f"  {n_:12s} {w_:+.4f}  [{lo:+.4f}, {hi:+.4f}]  ->", "different from 0" if (lo > 0 or hi < 0) else "NOT clearly different from 0")
print()
print("PROMPT OVERLAP check:")
for line in overlap_msg: print(line)
print("test split used: NO")
print("==========================================")

## Figures (train and validation data only; the test split is not used)
5 pictures, also saved as PNG files in `arena/results/figures` for the report.

In [ ]:
import matplotlib.pyplot as plt                     # matplotlib: the standard Python plotting library

FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)     # PNG copies go here (handy for the report)

# ---------- Figure 1: how much does lambda matter? ----------
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, style in [("log", "o-"), ("raw", "s--")]:
    lams = [r[0] for r in tune[name]]               # lambda values tried
    lls = [r[1] for r in tune[name]]                # validation log loss at each lambda
    for a in ax:
        a.plot(lams, lls, style, label=name + " features")
for a in ax:
    a.set_xscale("log"); a.set_xlabel("lambda (log scale)"); a.set_ylabel("validation log loss")
ax[0].set_title("Zoomed in: tiny changes only"); ax[0].legend()
ax[1].set_ylim(0.60, 0.70)                          # the same curves on an honest scale
ax[1].axhline(np.log(2), color="gray", ls=":", label="coin flip (0.6931)")
ax[1].set_title("Same data, honest scale: flat"); ax[1].legend()
plt.tight_layout(); plt.savefig(FIG / "fig1_lambda_curve.png", dpi=150); plt.show()

# ---------- Figure 2: accuracy of each model with 95% intervals (validation) ----------
names = ["baseline (longer wins)", "log features", "raw features", "log words only"]
stats = [session_boot(a, codes_va) for a in [c_base, c_log, c_raw, c_wds]]    # (mean, low, high) per model
fig, ax = plt.subplots(figsize=(7, 3.5))
for i, (m, lo, hi) in enumerate(stats):
    ax.errorbar(m, i, xerr=[[m - lo], [hi - m]], fmt="o", capsize=4)          # dot = accuracy, bar = 95% interval
ax.set_yticks(range(len(names))); ax.set_yticklabels(names); ax.invert_yaxis()
ax.set_xlabel("validation accuracy (dot) with 95% interval"); ax.set_title("Overlapping bars = no clear difference")
plt.tight_layout(); plt.savefig(FIG / "fig2_accuracy_intervals.png", dpi=150); plt.show()

# ---------- Figure 3: the learned weights with 95% intervals ----------
fig, ax = plt.subplots(figsize=(7, 3.5))
ypos = np.arange(len(LOGNAMES))
ax.errorbar(w_hat, ypos, xerr=[w_hat - w_lo, w_hi - w_hat], fmt="o", capsize=4)
ax.axvline(0, color="gray", ls=":")                 # a bar that does not touch this line = a real effect
ax.set_yticks(ypos); ax.set_yticklabels(LOGNAMES); ax.invert_yaxis()
ax.set_xlabel("weight (scaled units; positive = more of it helps a response win)"); ax.set_title("What the model learned")
plt.tight_layout(); plt.savefig(FIG / "fig3_weights.png", dpi=150); plt.show()

# ---------- Figure 4: are the probabilities trustworthy? (calibration, validation) ----------
p_va = sigmoid(M["log"]["z"][va] @ w_hat)           # predicted P(A wins) for validation votes
g = pd.DataFrame({"p": p_va, "y": y_all[va], "bin": pd.qcut(p_va, 10, duplicates="drop")}).groupby("bin", observed=True)
mp, my, mn = g.p.mean().to_numpy(), g.y.mean().to_numpy(), g.size().to_numpy()   # per group: mean predicted, real A-win rate, size
se = np.sqrt(my * (1 - my) / mn)                    # rough noise on each real rate
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0.3, 0.7], [0.3, 0.7], "k:", label="perfect")
ax.errorbar(mp, my, yerr=1.96 * se, fmt="o", capsize=3, label="model")
ax.set_xlabel("predicted P(A wins)"); ax.set_ylabel("actual share where A won"); ax.set_title("Calibration (10 equal groups)"); ax.legend()
plt.tight_layout(); plt.savefig(FIG / "fig4_calibration.png", dpi=150); plt.show()

# ---------- Figure 5: the length signal in the raw data (train + validation votes) ----------
mk = tr | va
lw = np.log1p(dec.a_words.to_numpy()) - np.log1p(dec.b_words.to_numpy())   # > 0 means A is longer than B
d5 = pd.DataFrame({"lw": lw[mk], "y": y_all[mk]})
d5["bin"] = pd.qcut(d5.lw, 20, duplicates="drop")   # 20 groups with equal numbers of votes
g5 = d5.groupby("bin", observed=True)
x5, y5, n5 = g5.lw.mean().to_numpy(), g5.y.mean().to_numpy(), g5.size().to_numpy()
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(x5, y5, yerr=1.96 * np.sqrt(y5 * (1 - y5) / n5), fmt="o", capsize=3)
ax.axhline(0.5, color="gray", ls=":"); ax.axvline(0, color="gray", ls=":")
ax.set_xlabel("log(1+words of A) minus log(1+words of B): right = A is longer")
ax.set_ylabel("share of votes where A won"); ax.set_title("Longer responses win more often, but the effect is modest")
plt.tight_layout(); plt.savefig(FIG / "fig5_length_signal.png", dpi=150); plt.show()

# STOP: Section 5B uses the test split
It is locked. Unlock it only after lambda is agreed from the validation results above. Run it once. Do not change lambda afterwards.
Your published run used lambda = 300 and gave: test accuracy 0.6028, test log loss 0.6631, and on the baseline's 7,693 votes model 0.6039 vs baseline 0.6007 (difference +0.0032, 95% interval [-0.0042, +0.0112]).

In [ ]:
# ---------- SECTION 5B: evaluate the test split ONCE, and save results ----------
CONFIRMED = False   # change to True only after you have seen REPORT A and agreed on lambda
assert CONFIRMED, "Section 5B is locked. Set CONFIRMED = True after agreeing on lambda."

# ----- Final model: the main (log) variant at the agreed lambda, fitted on TRAIN only -----
w_final = fit(M["log"], LAM)
ztr = M["log"]["z"]                                              # scaled differences for ALL votes
codes_te = pd.factorize(sess[mask_te])[0]                        # session numbers for the test votes the baseline can answer
p_te = sigmoid(ztr[mask_te] @ w_final); y_te = y_all[mask_te]    # predicted P(A wins) and true labels on those votes
c_te = ((p_te > 0.5) == (y_te == 1)).astype(float)               # model correct? per vote
c_base_te = base_correct[mask_te].astype(float)                  # baseline correct? per vote
acc_diff = session_boot(c_te - c_base_te, codes_te)              # (difference, 95% low, 95% high), resampling sessions

# ----- Everything worth keeping, in one dictionary (saved as global.json) -----
res = {
  "model_version": "global_v1",
  "date": str(datetime.date.today()),
  "features": LOGNAMES,
  "transform": "log(1+count) on 5 counts, refusal 0/1; scale only (root mean square on train), no intercept; trained on both A/B orders",
  "selection_rule": "lowest validation log loss over grid",
  "grid": GRID,
  "chosen_lambda": float(LAM),
  "validation": {"accuracy": acc(w_final, ztr[va], y_all[va]), "log_loss": logloss(w_final, ztr[va], y_all[va]), "n_decisive": int(va.sum())},
  "test": {"accuracy": acc(w_final, ztr[te], y_all[te]), "log_loss": logloss(w_final, ztr[te], y_all[te]), "n_decisive": int(te.sum())},
  "baseline_longer_words": {"validation_accuracy": float(base_correct[mask_va].mean()), "n_validation": int(mask_va.sum()),
                            "test_accuracy": float(c_base_te.mean()), "n_test": int(mask_te.sum())},
  "test_same_votes_as_baseline": {"model_accuracy": float(c_te.mean()), "difference_vs_baseline": acc_diff[0],
                                  "ci95": [acc_diff[1], acc_diff[2]]},
  "weights_scaled": [float(x) for x in w_final],
  "scale": [float(x) for x in M["log"]["scale"]],
}
(OUT / "global.json").write_text(json.dumps(res, indent=2))      # results file for the team and the report
# Model files for the later handoff to the demo / cloud (artifacts): the weights and the scaler.
G = OUT / "global_model"
G.mkdir(exist_ok=True)
np.save(G / "w_global.npy", w_final)
(G / "scaler.json").write_text(json.dumps({"feature_names": LOGNAMES, "scale": [float(x) for x in M["log"]["scale"]]}, indent=2))

print("================ REPORT B ================")
print("lambda used:", LAM)
print("VALIDATION (all decisive):", {k: round(v, 4) for k, v in res["validation"].items() if k != "n_decisive"}, "n =", res["validation"]["n_decisive"])
print("TEST (all decisive):      ", {k: round(v, 4) for k, v in res["test"].items() if k != "n_decisive"}, "n =", res["test"]["n_decisive"])
print(f"TEST, same {res['baseline_longer_words']['n_test']} votes as baseline: model {res['test_same_votes_as_baseline']['model_accuracy']:.4f} vs baseline {res['baseline_longer_words']['test_accuracy']:.4f}")
print(f"   difference {acc_diff[0]:+.4f}  95% interval [{acc_diff[1]:+.4f}, {acc_diff[2]:+.4f}]")
print("saved:", OUT / "global.json", "and", G)
print("==========================================")